# M3 step 5: setup check (task 1)

Before any new test, this notebook confirms that the model, the code and the 500 test photos on this
machine give the same results as notebook 4. It repeats notebook 4's test exactly (same edits, same
random payloads from seed 0), so notebook 4 itself does not need to be edited.

If the table below does not match, stop and tell Rishitha before running anything else.

**GPU:** not needed. On a CPU this takes a few minutes.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

## Repeat notebook 4's test on all 500 photos

In [ ]:
files = E.list_photos(S.TEST_DIR, S.N_TEST_PHOTOS)
print('Test photos:', len(files))

EDITS = {
    'PNG': lambda im: E.save_as(im, 'PNG'),
    'JPEG 90': lambda im: E.save_as(im, 'JPEG', 90),
    'JPEG 70': lambda im: E.save_as(im, 'JPEG', 70),
    'JPEG 50': lambda im: E.save_as(im, 'JPEG', 50),
    'Resized to 224': lambda im: E.save_as(im.resize((224, 224), Image.BILINEAR), 'PNG'),
}
rng = np.random.default_rng(0)                 # notebook 4's seed and order
bits_ok = {name: [] for name in EDITS}
verdict_ok = {name: [] for name in EDITS}
valid_any = {name: [] for name in EDITS}       # not in notebook 4: needed for the invalidation rate
psnr, unstamped_valid = [], 0
for path in E.tqdm(files):
    photo = Image.open(path).convert('RGB')
    verdict = ['Real', 'AI', 'Uncertain'][rng.integers(3)]
    p_ai, tag = float(rng.random()), int(rng.integers(2**14))
    expected = mw.encode_payload(verdict, p_ai, image_tag=tag)
    stamped = mw.embed(m3, photo, verdict, p_ai, image_tag=tag)
    psnr.append(E.psnr(E.save_as(stamped, 'PNG'), photo))
    unstamped_valid += mw.extract(m3, E.save_as(photo, 'PNG'))['valid']
    for name, edit in EDITS.items():
        bits = mw.read_bits(m3, edit(stamped))
        bits_ok[name].append(np.mean([a == b for a, b in zip(bits, expected)]))
        decoded = mw.decode_payload(bits)
        verdict_ok[name].append(decoded['valid'] and decoded['verdict'] == verdict)
        valid_any[name].append(decoded['valid'])

## Compare with the handover guide

In [ ]:
GUIDE = {'PNG': (99.85, 96.0), 'JPEG 90': (99.83, 95.2), 'JPEG 70': (99.58, 90.0),
         'JPEG 50': (98.75, 72.4), 'Resized to 224': (99.83, 95.4)}
rows = []
for name in EDITS:
    b, v = 100 * np.mean(bits_ok[name]), 100 * np.mean(verdict_ok[name])
    gb, gv = GUIDE[name]
    # Small differences can come from a different GPU or library version; large ones mean a problem.
    rows.append({'edit': name, 'bit_accuracy_%': round(b, 2), 'guide_bits_%': gb,
                 'valid_verdict_%': round(v, 2), 'guide_valid_%': gv,
                 'invalidation_rate_%': round(100 - 100 * np.mean(valid_any[name]), 2),
                 'match': abs(b - gb) <= 0.15 and abs(v - gv) <= 1.0})
check = pd.DataFrame(rows)
print(check.to_string(index=False))
print(f'PSNR at full size: {np.mean(psnr):.2f} dB (guide 31.50)')
print(f'Unstamped photos read as valid: {unstamped_valid} of {len(files)} (guide 0)')
ok = bool(check['match'].all()) and abs(np.mean(psnr) - 31.50) <= 0.05 and unstamped_valid == 0
print('\nSETUP OK: results match notebook 4.' if ok else '\nMISMATCH: stop and tell Rishitha.')

In [ ]:
E.save_results(S.RESULTS_DIR, 'setup_check', summary=None,
               meta={'table': rows, 'psnr': float(np.mean(psnr)), 'unstamped_valid': int(unstamped_valid),
                     'ok': ok, 'model_epoch': m3.epoch, 'model_sha256': E.file_sha256(S.MODEL_PATH),
                     'environment': E.environment_info()})